In [6]:
import re, os, time, requests
import pandas as pd
import folium
from IPython.display import display, HTML, clear_output
from ipywidgets import Text, Dropdown, IntSlider, Button, HBox, VBox, Output, Checkbox

# Selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# === API KEYS ===
KAKAO_API_KEY = "4f25ea2e2ce1c7a3a76f4ed96a3a44ff"
NAVER_CLIENT_ID = "ljQKyv2MAkTex1q2Vjeg"
NAVER_CLIENT_SECRET = "TjKu5MEvWH"

# =========================
# 내 위치 (IP 기반)
# =========================
def get_ip_coords():
    try:
        r = requests.get("https://ipinfo.io/json").json()
        lat, lon = map(float, r["loc"].split(","))
        return lat, lon
    except:
        return 37.5665, 126.9780  # fallback: 서울시청

# =========================
# 주소 → 좌표 변환
# =========================
def get_coords_from_address(address):
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        r = requests.get(url, headers=headers, params={"query": address}, timeout=7).json()
        if r.get("documents"):
            return float(r["documents"][0]["y"]), float(r["documents"][0]["x"])
    except:
        pass
    return None, None

# =========================
# 이미지 가져오기 (네이버/카카오)
# =========================
def get_naver_image(query):
    url = "https://openapi.naver.com/v1/search/image"
    headers = {"X-Naver-Client-Id": NAVER_CLIENT_ID,
               "X-Naver-Client-Secret": NAVER_CLIENT_SECRET}
    params = {"query": query, "display": 1, "sort": "sim"}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=7)
        r.raise_for_status()
        res = r.json()
        if res.get("items"):
            return res["items"][0]["link"]
    except:
        pass
    return None

def get_og_image_from_kakao(place_url):
    try:
        h = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(place_url, headers=h, timeout=7)
        html = r.text
        m = re.search(r'property=["\']og:image["\'][^>]+content=["\']([^"\']+)["\']', html)
        if m:
            return m.group(1)
    except:
        pass
    return None

# =========================
# Selenium 드라이버 & 평점 크롤링
# =========================
def init_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def _extract_first_float(text):
    m = re.search(r'(\d+(?:\.\d+)?)', text)
    return float(m.group(1)) if m else None

def get_kakao_ratings_bulk(driver, place_urls):
    results = {}
    selectors = ["span.num_star","span.txt_score","em.num_rate","strong.score_num"]
    for url in place_urls:
        rating = None
        try:
            driver.get(url)
            WebDriverWait(driver, 5).until(lambda d: d.execute_script("return document.readyState")=="complete")
            for sel in selectors:
                try:
                    elem = driver.find_element(By.CSS_SELECTOR, sel)
                    rating = _extract_first_float(elem.text)
                    if rating: break
                except:
                    continue
        except:
            pass
        results[url] = rating
    return results

# =========================
# 카카오 장소 검색
# =========================
def get_places(category, lat, lon, size=15, sort="accuracy", radius=3000):
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    
    # 카테고리 → 코드 매핑
    category_codes = {
        "헬스장": "CT1",   # 문화시설 (헬스장은 정확 코드 없음 → 문화시설로 근접 검색)
        "공원": "AT4",    # 관광명소
        "내과": "HP8",    # 병원
        "종합병원": "HP8",
        "내분비내과": "HP8"
    }
    code = category_codes.get(category, "HP8")

    params = {
        "category_group_code": code,
        "x": lon, "y": lat,
        "radius": radius,
        "size": size,
        "sort": "distance" if sort=="distance" else "accuracy"
    }
    
    try:
        res = requests.get(url, headers=headers, params=params, timeout=7).json()
    except:
        return pd.DataFrame()

    rows = []
    for p in res.get("documents", []):
        addr_q = p["road_address_name"] or p["address_name"]
        rows.append({
            "이름": p["place_name"], "카테고리": category,
            "주소": addr_q, "위도": float(p["y"]), "경도": float(p["x"]),
            "전화번호": p["phone"], "링크": p["place_url"]
        })
    return pd.DataFrame(rows)


# =========================
# Top5 카드 HTML
# =========================
def top5_cards_html(df):
    if df.empty: return "<p style='color:#777'>표시할 결과가 없습니다.</p>"
    html = "<h3>📌 Top 5</h3>"
    for _, r in df.head(5).iterrows():
        img_tag = f"<img src='{r['대표이미지']}' style='width:100%;height:120px;object-fit:cover;border-radius:8px;'>" if r['대표이미지'] else ""
        rating_txt = f"{r['평점']:.1f}" if "평점" in r and pd.notna(r["평점"]) else "정보 없음"
        html += f"""
        <div style="width:220px;border:1px solid #ddd;border-radius:10px;padding:10px;margin-bottom:10px;">
            {img_tag}
            <h4>{r['이름']}</h4>
            <p>카테고리: {r['카테고리']}</p>
            <p>⭐ 평점: {rating_txt}</p>
            <p style="font-size:12px;color:#777;">{r['주소']}</p>
        </div>
        """
    return html

# =========================
# 지도 생성 & 표시
# =========================
def build_and_show(address, category, sort, radius_km, use_my_location=False):
    if use_my_location:
        lat, lon = get_ip_coords()
        address = "내 위치 기반"
    else:
        lat, lon = get_coords_from_address(address)
        if not lat: lat, lon = 37.5665, 126.9780

    m = folium.Map(location=[lat, lon], zoom_start=14)
    folium.Marker([lat, lon], popup=f"<b>내 위치</b><br>{address}",
                  icon=folium.Icon(color="red", icon="home")).add_to(m)

    dfs = []
    cats = ["헬스장","공원","내과","종합병원","내분비내과"] if category=="전체" else [category]
    for cat in cats:
        dfs.append(get_places(cat, lat, lon, radius=radius_km*1000, sort=sort))
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    if not df.empty:
        driver = init_driver()
        try:
            ratings = get_kakao_ratings_bulk(driver, df["링크"].tolist())
            df["평점"] = df["링크"].map(ratings)
        finally:
            driver.quit()
        if sort=="rating": df = df.sort_values("평점", ascending=False)

        for row in df.itertuples():
            rating_txt = f"{row.평점:.1f}" if pd.notna(getattr(row,"평점")) else "정보 없음"
            popup = folium.Popup(f"<b>{row.이름}</b><br>주소: {row.주소}<br>⭐ {rating_txt}", max_width=250)
            folium.Marker([row.위도,row.경도],popup=popup).add_to(m)

    os.makedirs("./_maps", exist_ok=True)
    map_path = f"./_maps/map_{int(time.time())}.html"
    m.save(map_path)

    return f"""
    <div style="display:flex;gap:20px;">
      <div style="flex:2;"><iframe src="{map_path}" style="width:100%;height:600px;border:none;"></iframe></div>
      <div style="flex:1;max-width:280px;overflow:auto;height:600px;">{top5_cards_html(df)}</div>
    </div>
    """

# =========================
# UI 위젯
# =========================
address_input   = Text(value="서울 종로구 세종대로 110", description="주소:", layout={'width':'60%'})
use_my_location = Checkbox(value=False, description="내 위치 자동 사용")
category_dd     = Dropdown(options=["전체","헬스장","공원","내과","종합병원","내분비내과"], value="전체", description="보기:")
sort_dd         = Dropdown(options=[("정확도순","accuracy"),("가까운순","distance"),("평점순","rating")], value="accuracy", description="정렬:")
radius_slider   = IntSlider(value=3, min=1, max=20, step=1, description="반경(km):")
btn_search      = Button(description="검색", button_style="success")
out             = Output()

def on_click(b):
    with out:
        clear_output()
        html = build_and_show(address_input.value, category_dd.value, sort_dd.value, radius_slider.value, use_my_location.value)
        display(HTML(html))

ui = VBox([HBox([address_input, use_my_location, btn_search]), category_dd, sort_dd, radius_slider, out])
btn_search.on_click(on_click)
display(ui)
